# Optional MLflow Tracking and Model Registry quickstart

This notebook is an additive ML example. It uses the standalone MLflow service and leaves the core lakehouse resources unchanged. Start the lakehouse with Jupyter, then start the MLflow Compose add-on as documented in the repository README.

Open this notebook in the JupyterLab server at http://localhost:8888/lab. If using VS Code, connect to that Jupyter server and select its Python 3 (ipykernel) kernel; do not select a local kernel such as data_eng_env.

The first cell installs the MLflow client and scikit-learn into the selected Jupyter kernel. Re-run it after recreating the container.

In [3]:
%pip install mlflow==3.17.2 scikit-learn

In [5]:
import mlflow.sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from IPython.display import Markdown, display

import mlflow

# Containers on lakedemo reach the standalone tracking server by service name.
mlflow.set_tracking_uri("http://mlflow:5000")
experiment = mlflow.set_experiment("lakehouse-ml-quickstart")

X, y = make_regression(n_samples=300, n_features=4, noise=12, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = LinearRegression()
model.fit(X_train, y_train)
rmse = mean_squared_error(y_test, model.predict(X_test)) ** 0.5

with mlflow.start_run() as run:
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("n_features", X.shape[1])
    mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        registered_model_name="lakehouse-demo-regressor",
    )
    print(f"Logged run {run.info.run_id}; RMSE={rmse:.3f}")

experiment_url = f"http://localhost:5000/#/experiments/{experiment.experiment_id}"
display(Markdown(f"[Open the experiment in MLflow]({experiment_url})"))

After the model run completes, use the clickable experiment link printed by the code cell. It opens the host-accessible MLflow UI at `localhost:5000` and the exact experiment ID. The registered model appears under **Models**. The run metadata is in MLflow's dedicated PostgreSQL volume and model files are in its dedicated artifact volume.